# NB03 · Semana 7 — DuckDB y Delta Lake sobre el lake

**Ingeniería de Datos — LCD · UCA Rosario · 2026 · grupo07**

## Objetivo

Consultar el bronze (Delta Lake en MinIO) con **DuckDB**, explorarlo para encontrar los problemas que tiene que limpiar silver, y ver el **historial de versiones** de Delta (*time travel*).

Cada hallazgo de la sección 2 termina en una decisión de `stg_urlhaus` (ver `docs/entregas/E2_dbt.md`).

In [1]:
# === Configuración del entorno — NO tocar ===
import os, sys
_d = os.getcwd()
while True:
    if os.path.exists(os.path.join(_d, "entorno.py")):
        sys.path.insert(0, _d); break
    _p = os.path.dirname(_d)
    if _p == _d:
        raise FileNotFoundError("No encuentro entorno.py subiendo desde el directorio actual.")
    _d = _p
from entorno import BUCKET, BRONZE, STORAGE_OPTIONS, info
info()

Entorno detectado: servidor de la cátedra  (grupo: grupo07)
  Lake / bucket   : grupo07   →   bronze en s3://grupo07/bronze/series
  PostgreSQL      : base 'warehouse', usuario 'grupo07', host 'postgres'
  Material en     : /home/jovyan/work


In [2]:
# Dependencias: el JupyterHub puede reiniciarse y perder lo instalado con pip
%pip install -q -r {os.path.join(_d, 'requirements.txt')}

Note: you may need to restart the kernel to use updated packages.


## 1. Leer el bronze

`deltalake` lee la versión vigente de la tabla (solo los archivos que la componen) y DuckDB la consulta con SQL, sin cargarla en ninguna base.

In [3]:
import duckdb
import pandas as pd
from deltalake import DeltaTable

dt = DeltaTable(BRONZE, storage_options=STORAGE_OPTIONS)
bronze = dt.to_pyarrow_table()
con = duckdb.connect()
con.sql("set TimeZone = 'UTC'")          # las fechas de URLhaus están en UTC
con.register("bronze", bronze)
pd.set_option("display.max_colwidth", 80)

print(f"{BRONZE}: versión {dt.version()}, {bronze.num_rows} filas, {bronze.num_columns} columnas")
print(bronze.schema)

s3://grupo07/bronze/series: versión 213, 189785 filas, 9 columnas
id: int64
date_added: timestamp[us, tz=UTC]
url: string
url_status: string
last_online: string
threat: string
tags: string
urlhaus_link: string
reporter: string


## 2. Exploración: qué problemas tiene el dato crudo

### 2.1 ¿Hay URLs repetidas o sin datos clave?

In [4]:
con.sql("""
    select count(*)                                   as filas,
           count(distinct id)                         as ids_distintos,
           count(*) - count(distinct id)              as filas_repetidas,
           count(*) filter (where id is null)         as sin_id,
           count(*) filter (where date_added is null) as sin_fecha,
           count(*) filter (where tags is null or trim(tags) = '') as sin_tags,
           min(date_added) as primera, max(date_added) as ultima
    from bronze
""").df()

,filas,ids_distintos,filas_repetidas,sin_id,sin_fecha,sin_tags,primera,ultima
0,189785,189785,0,0,0,1945,2018-06-05 22:09:14+00:00,2026-11-18 22:02:06+00:00


→ **Silver deduplica por `id`** y descarta filas sin id o sin fecha, por si una reingesta repite URLs. Las que no tienen etiquetas quedan como `sin_familia` / `desconocido`.

### 2.2 ¿Sirve `threat` para distinguir amenazas?

In [5]:
con.sql("""
    select threat, count(*) as urls, round(100.0 * count(*) / sum(count(*)) over (), 2) as pct
    from bronze group by 1 order by 2 desc
""").df()

,threat,urls,pct
0,malware_download,189783,100.0
1,c2,2,0.0


→ Un solo valor, `malware_download`: **no distingue nada**. Por eso silver deduce dos dimensiones nuevas de las etiquetas: la **familia de malware** y el **tipo de archivo**.

### 2.3 ¿Qué hay en las etiquetas?

In [6]:
etiquetas = con.sql(r"""
    with t as (select trim(unnest(string_split(regexp_replace(coalesce(tags, ''), '[\[\]''"]', '', 'g'), ','))) as tag
               from bronze)
    select tag, count(*) as urls from t where tag <> '' group by 1 order by 2 desc limit 40
""").df()
etiquetas

,tag,urls
0,github,113659
1,zip,68685
2,SmartLoader,59024
3,zapret,49152
4,SalatStealer,33777
5,elf,23429
6,Mozi,20523
7,mirai,18709
8,LuaJIT-loader,18387
9,ua-wget,12881


Las etiquetas mezclan la familia (`Mozi`, `mirai`, `LuaJIT-loader`) con el tipo de archivo (`elf`, `exe`, `zip`), la arquitectura (`32-bit`, `mips`, `arm`), el lugar donde estaba alojado (`github`, `discord`), el país y etiquetas de quien reporta (`dropped-by-*`, `ua-*`). Y aparecen cosas peores:

In [7]:
con.sql(r"""
    with t as (select trim(unnest(string_split(regexp_replace(coalesce(tags, ''), '[\[\]''"]', '', 'g'), ','))) as tag
               from bronze)
    select case
             when regexp_matches(tag, '^[0-9]{1,3}([.-][0-9]{1,3}){3}([.:-][0-9]{1,5})?$') then 'IP'
             when regexp_matches(lower(tag), '[.-](com|net|org|xyz|top|site|space|to|ru|cn)$') then 'dominio'
             when regexp_matches(lower(tag), '^[0-9a-f]{6,}$') then 'pedazo de hash'
             when lower(tag) in ('cowrie', 'honeypot', 'ssh', 'telnet') then 'honeypot / protocolo'
           end as tipo_de_ruido,
           count(distinct tag) as etiquetas_distintas, count(*) as urls,
           any_value(tag) as ejemplo
    from t
    group by 1 having tipo_de_ruido is not null order by 3 desc
""").df()

,tipo_de_ruido,etiquetas_distintas,urls,ejemplo
0,IP,772,5229,130-12-181-99
1,honeypot / protocolo,4,2555,cowrie
2,dominio,31,391,aura-tcplegit-space
3,pedazo de hash,39,372,54e64e


→ **Silver toma como familia la primera etiqueta que no es ninguna de esas cosas.** Las IPs, los dominios, los honeypots y los hashes los encontramos recién con datos reales, cuando aparecieron como "familias" en el tablero. El test `assert_familia_no_es_tag_generico` evita que vuelvan.

### 2.4 ¿La misma familia aparece escrita distinto?

In [8]:
con.sql(r"""
    with t as (select trim(unnest(string_split(regexp_replace(coalesce(tags, ''), '[\[\]''"]', '', 'g'), ','))) as tag
               from bronze)
    select lower(tag) as familia, list(distinct tag) as grafias, count(*) as urls
    from t where tag <> ''
    group by 1 having count(distinct tag) > 1
    order by 3 desc limit 10
""").df()

,familia,grafias,urls
0,elf,"[elf, ELF]",23432
1,stealer,"[stealer, Stealer]",635
2,dropper,"[dropper, Dropper]",617
3,vscode,"[VSCode, vscode]",514
4,macos,"[macOS, macos]",169
5,amos,"[Amos, amos]",79
6,miner,"[miner, Miner]",21
7,nodejs,"[NodeJS, nodeJS]",21
8,yogi,"[yogi, Yogi]",18
9,react2shell,"[react2shell, React2Shell]",9


→ `Mozi` / `mozi`, `Mirai` / `mirai`, ... **Silver unifica con la grafía más frecuente**, si no una misma familia se partiría en dos en el ranking.

### 2.5 ¿Cuántas URLs hay por mes? (cobertura)

In [9]:
por_mes = con.sql("""
    select strftime(date_added, '%Y-%m') as mes, count(*) as urls
    from bronze group by 1 order by 1
""").df()
print(por_mes.tail(14).to_string(index=False))
print(f"\nMeses con datos: {len(por_mes)} · meses con menos de 1.000 URLs: {(por_mes.urls < 1000).sum()}")

    mes   urls
2025-10   1072
2025-11    148
2025-12     79
2026-01    145
2026-02    168
2026-03    248
2026-04    314
2026-05     75
2026-06   4748
2026-07   4598
2026-08  15663
2026-09  15203
2026-10 122726
2026-11  20564

Meses con datos: 93 · meses con menos de 1.000 URLs: 85


→ **Antes de agosto de 2026 los meses tienen muchas menos URLs:** el export de URLhaus conserva bien solo los últimos uno o dos meses y ya no publica las URLs viejas. Es un hueco del origen, no del pipeline, y está documentado en la governance. El salto de octubre, en cambio, es real: un pico de campañas (ver el tablero).

### 2.6 ¿Quién reporta?

In [10]:
con.sql("""
    select reporter, count(*) as urls, round(100.0 * count(*) / sum(count(*)) over (), 1) as pct,
           min(date_added)::date as desde, max(date_added)::date as hasta
    from bronze group by 1 order by 2 desc limit 8
""").df()

,reporter,urls,pct,desde,hasta
0,wristhulk,96956,51.1,2026-09-29,2026-11-18
1,shmulc,15227,8.0,2026-10-06,2026-10-18
2,adliwahid,15074,7.9,2026-03-20,2026-11-18
3,geenensp,10631,5.6,2022-02-19,2026-11-18
4,cesnet_certs,6704,3.5,2024-08-30,2026-11-18
5,anonymous,6419,3.4,2018-08-02,2026-11-18
6,BlinkzSec,6275,3.3,2024-05-30,2026-11-18
7,abuse_ch,5990,3.2,2018-09-23,2026-11-18


→ Unos pocos reporteros cargan buena parte de las URLs, a veces miles en pocos días. **Un pico en el tablero puede ser un reportero muy activo y no una campaña nueva**: queda como limitación en la governance. Además, `reporter` identifica a una persona o equipo: se queda en silver y no se publica.

## 3. Historial de versiones (*time travel*)

Delta guarda en su log cada operación sobre la tabla, y en principio permite leer cualquier versión anterior.

In [11]:
historia = pd.DataFrame(dt.history(limit=10))
historia["timestamp"] = pd.to_datetime(historia["timestamp"], unit="ms", utc=True)
historia[[c for c in ["version", "timestamp", "operation", "operationParameters"] if c in historia.columns]]

,version,timestamp,operation,operationParameters
0,213,2026-11-18 22:07:06.364000+00:00,WRITE,{'mode': 'Append'}
1,212,2026-11-18 21:54:41.447000+00:00,VACUUM END,{'status': 'COMPLETED'}
2,211,2026-11-18 21:54:41.427000+00:00,VACUUM START,"{'specifiedRetentionMillis': '0', 'defaultRetentionMillis': '604800000', 're..."
3,210,2026-11-18 21:54:41.378000+00:00,MERGE,"{'matchedPredicates': '[{""actionType"":""update""}]', 'notMatchedBySourcePredic..."
4,209,2026-11-18 21:53:40.554000+00:00,VACUUM END,{'status': 'COMPLETED'}
5,208,2026-11-18 21:53:40.531000+00:00,VACUUM START,"{'retentionCheckEnabled': 'false', 'specifiedRetentionMillis': '0', 'default..."
6,207,2026-11-18 21:53:40.485000+00:00,MERGE,"{'matchedPredicates': '[{""actionType"":""update""}]', 'mergePredicate': 't.id =..."
7,206,2026-11-18 20:40:08.262000+00:00,VACUUM END,{'status': 'COMPLETED'}
8,205,2026-11-18 20:40:08.241000+00:00,VACUUM START,"{'retentionCheckEnabled': 'false', 'defaultRetentionMillis': '604800000', 's..."
9,204,2026-11-18 20:40:08.196000+00:00,MERGE,"{'notMatchedBySourcePredicates': '[]', 'matchedPredicates': '[{""actionType"":..."


**Las versiones viejas de bronze ya no se pueden leer.** Después de escribir, `extraer_bronze` hace `vacuum` con retención 0, porque dbt lee los `.parquet` directamente y sin `vacuum` leería también archivos que ya no son parte de la tabla. Eso borra los archivos de las versiones anteriores: el log las recuerda, pero sus datos ya no están. Lo sufrimos el 18/11/2026, cuando un reproceso borró URLs y el *time travel* no tenía a dónde volver (ver `docs/INCIDENTE_2026-11-18_backfill.md`).

In [12]:
anterior = max(dt.version() - 1, 0)
try:
    n = DeltaTable(BRONZE, version=anterior, storage_options=STORAGE_OPTIONS).to_pyarrow_table(columns=["id"]).num_rows
    print(f"Versión {anterior}: {n} filas")
except Exception as e:
    print(f"La versión {anterior} no se puede leer: {type(e).__name__}. Sus archivos los borró el vacuum.")

Versión 212: 189756 filas


Para ver el *time travel* funcionando, una tabla de demostración aparte (no toca bronze): se escribe dos veces y se lee la versión 0.

In [13]:
from deltalake import write_deltalake
import pyarrow as pa

DEMO = f"s3://{BUCKET}/demo/time_travel"
write_deltalake(DEMO, pa.table({"id": [1, 2, 3], "estado": ["online"] * 3}),
                mode="overwrite", storage_options=STORAGE_OPTIONS)
write_deltalake(DEMO, pa.table({"id": [4, 5], "estado": ["offline"] * 2}),
                mode="append", storage_options=STORAGE_OPTIONS)

actual = DeltaTable(DEMO, storage_options=STORAGE_OPTIONS)
inicio = actual.version() - 1
v0 = DeltaTable(DEMO, version=inicio, storage_options=STORAGE_OPTIONS)
print(f"Versión actual ({actual.version()}): {actual.to_pyarrow_table().num_rows} filas")
print(f"Versión {inicio} (antes del append): {v0.to_pyarrow_table().num_rows} filas")

Versión actual (1): 5 filas
Versión 0 (antes del append): 3 filas


## Cierre

DuckDB sobre Delta permite explorar el lake con SQL sin moverlo a una base. Lo que encontramos acá (un solo `threat`, etiquetas con ruido, grafías distintas, huecos de cobertura, reporteros dominantes) es lo que limpia silver y lo que documenta la governance. Pendiente de mejora: si dbt leyera la tabla Delta con `delta_scan` en vez de los `.parquet` sueltos, se podría guardar historia de versiones (sacar el `vacuum` con retención 0) y tener *time travel* real sobre bronze.